In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Summarize messages

In [2]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [3]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user's primary goal is to gather information about Lunapolis, the fictional capital of the moon, including its weather, population, and socio-economic conditions.\n\n## SUMMARY\nThe capital of the moon is identified as Lunapolis. Current weather conditions in Lunapolis include clear skies with a high temperature of 120°C and a low of -100°C. The population of cheese miners in Lunapolis is reported to be 100,000. Concerns were expressed regarding the cheese miners' union potentially striking due to dissatisfaction with the new president.\n\n## ARTIFACTS\nNone.\n\n## NEXT STEPS\nNone.", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='b594966e-cab4-478e-abb6-9d9a089d8b48'),
              HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?", additional_kwargs={}, response_metadata={}, id='cb2feb0b-

In [4]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT
The user's primary goal is to gather information about Lunapolis, the fictional capital of the moon, including its weather, population, and socio-economic conditions.

## SUMMARY
The capital of the moon is identified as Lunapolis. Current weather conditions in Lunapolis include clear skies with a high temperature of 120°C and a low of -100°C. The population of cheese miners in Lunapolis is reported to be 100,000. Concerns were expressed regarding the cheese miners' union potentially striking due to dissatisfaction with the new president.

## ARTIFACTS
None.

## NEXT STEPS
None.


## Trim/delete messages

In [5]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [6]:
agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [7]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='67d037e7-78fd-443f-9731-a926855788dd'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='21dfcd1d-e91c-4a12-b538-898802a15c71', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='491ace99-4720-4939-b07d-b3bc12691c74'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='e113bc18-c7d9-49d7-b592-0c2ea94496b7', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='038c2311-a328-4002-896e-84a2eaf53180'),
              AIMessage(content='I can’t read your device’s internal temperature from here, and

In [8]:
print(response["messages"][-1].content)

I can’t read your device’s internal temperature from here, and if it won’t turn on you won’t be able to get an accurate temperature reading anyway. If you’re worried about overheating, here are quick steps to troubleshoot and check safely:

- Check for heat symptoms now: is the exterior very hot to the touch, or does it have a burning smell? If it’s hot, power it off and let it cool in a ventilated area before trying again.
- Power supply basics:
  - Try plugging the charger into a different wall outlet.
  - If you have another compatible charger, try it to rule out a bad power adapter.
  - Look for any charger LED indicators; if there’s none, the charger or outlet might be at fault.
- Peripherals and battery:
  - unplug all peripherals (USB drives, HDMI, etc.).
  - If it’s a laptop with a removable battery, remove the battery, plug in AC power, and try to boot with no battery.
- Power cycle (soft reset):
  - With the device unplugged and (if possible) battery removed, hold the power b